In [19]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Load input files
# ---------------------------------------------------------

BASE_DIR = Path.cwd().parents[0]  # parent folder of 05-Final csv

weather_path = BASE_DIR / "02-Get weather" / "sites_weather_summary.csv"
hotels_path  = BASE_DIR / "03-booking_scraper" / "hotels.csv"

df_weather = pd.read_csv(weather_path)
df_hotels  = pd.read_csv(hotels_path)

print("Weather rows:", len(df_weather))
print("Hotels rows:", len(df_hotels))

Weather rows: 35
Hotels rows: 105


In [20]:
# ---------------------------------------------------------
# 2. Address cleaner
# ---------------------------------------------------------

def clean_address(addr: str) -> str:
    if not isinstance(addr, str) or addr.strip() == "":
        return addr
    # Keep only the first occurrence of "... France"
    if "France" in addr:
        before, _, _ = addr.partition("France")
        return (before.strip() + " France").strip()
    return addr.strip()

df_hotels["clean_address"] = df_hotels["address"].apply(clean_address)


In [21]:
# ---------------------------------------------------------
# 3. Prepare the merge
# ---------------------------------------------------------

# Normalise join key -> same logic as the spider
def norm_key(s: str) -> str:
    import unicodedata, re
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

df_weather["join_key"] = df_weather["site"].astype(str).apply(norm_key)
df_hotels["join_key"]    = df_hotels["destination"].astype(str).apply(norm_key)


In [22]:
# ---------------------------------------------------------
# 4. Merge: duplicate weather info for each hotel
# ---------------------------------------------------------

df_final = df_hotels.merge(df_weather, on="join_key", how="left", suffixes=("_hotel", "_site"))
df_final = df_final.drop(columns=["join_key"])

# Remove duplicate weather ID column (we keep only site_id from hotels)
df_final = df_final.drop(columns=["id"])

# Remove the dirty address and keep only the cleaned version
df_final = df_final.drop(columns=["address"])
df_final = df_final.rename(columns={"clean_address": "address"})

# Remove the duplicate site name from weather file
df_final = df_final.drop(columns=["site"])


In [23]:
# ---------------------------------------------------------
# 5. Save final output
# ---------------------------------------------------------

output_dir = BASE_DIR / "05-Final csv"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "Kayak_final.csv"

df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Final CSV written to:", output_path)
print("Rows in final file:", len(df_final))


Final CSV written to: c:\Users\smarg\OneDrive\Studies\Jedha\0-Fullstack\Project2-Kayak\05-Final csv\Kayak_final.csv
Rows in final file: 105
